In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [4]:
model = ChatGroq(
    model="llama-3.1-8b-instant",   # or llama3-70b-8192
    temperature=0.2
)

In [19]:
# state definition
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: str

In [ ]:
# create node

def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm to create outline

    prompt = f"Create a detailed outline for a blog post titled '{title}'."
    outline = model.invoke(prompt).content

    # update state and return
    state['outline'] = outline
    return state

In [21]:
# create node 

def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a comprehensive blog based on the title - {title} using the following outline \n{outline}'
    content = model.invoke(prompt).content
    state['content'] = content
    return state


In [22]:
# create node

def evaluate_blog(state: BlogState) -> BlogState:

    content = state['content']
    outline = state['outline']
    prompt = f'Based on this outline: {outline}, evaluate the following blog content for completeness and coherence: {content}. Generate an indegeous score.'
    evaluation = model.invoke(prompt).content
    state['evaluation'] = evaluation
    return state


In [ ]:
# graph definition

graph = StateGraph(BlogState)

# nodes

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', 'create_outline')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [24]:
initial_state = {'title': 'The Future of Artificial Intelligence in Everyday Life'}
final_state =  workflow.invoke(initial_state)

print(final_state)

{'title': 'The Future of Artificial Intelligence in Everyday Life', 'outline': "**I. Introduction**\n\n* Brief overview of the current state of Artificial Intelligence (AI)\n* Importance of AI in everyday life\n* Thesis statement: The future of AI in everyday life holds immense potential for transformation and improvement, but also raises concerns about its impact on society.\n\n**II. Current Applications of AI in Everyday Life**\n\n* Overview of existing AI applications:\n\t+ Virtual assistants (e.g. Siri, Alexa, Google Assistant)\n\t+ Image and speech recognition\n\t+ Personalized recommendations (e.g. Netflix, Amazon)\n\t+ Self-driving cars and drones\n\t+ Healthcare and medical diagnosis\n* Examples of AI-powered products and services\n\n**III. Emerging Trends in AI**\n\n* **Machine Learning (ML) and Deep Learning (DL)**: Explanation of these techniques and their applications\n* **Natural Language Processing (NLP)**: Discussion of its potential in human-computer interaction\n* **Co

In [25]:
print(final_state['evaluation'])

**Completeness Score: 8.5/10**

The provided content covers a wide range of topics related to Artificial Intelligence (AI) in everyday life, including its current applications, emerging trends, future predictions, and concerns. However, there are some areas that could be improved or expanded upon:

1. **Lack of depth in certain topics**: Some sections, such as the discussion on Machine Learning (ML) and Deep Learning (DL), are brief and could benefit from more detailed explanations.
2. **Insufficient examples**: While the content provides some examples of AI-powered products and services, more concrete examples could be used to illustrate the applications and benefits of AI.
3. **Limited discussion on ethics and regulation**: The section on concerns and challenges touches on bias, security, and regulation, but a more in-depth discussion on the ethics of AI and the need for regulation could be beneficial.
4. **No clear call to action**: The conclusion could be strengthened by including 